### Before you use the generate coco annotation, you must to verify it is right or not.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

image_path = Path("../assets/giftbox_task/images")
annotation_file = Path("../assets/giftbox_task/annotations/annotations.json")

images_list = sorted(image_path.glob("*.png"))[:10]  # Limit to 10 images for testing

# Load COCO annotation JSON
with open(annotation_file, "r") as f:
    coco_data = json.load(f)

# Build lookup: file_name -> image_id
image_id_map = {img["file_name"]: img["id"] for img in coco_data["images"]}
# Build lookup: image_id -> list of annotations
anns_by_image_id = {}
for ann in coco_data["annotations"]:
    anns_by_image_id.setdefault(ann["image_id"], []).append(ann)
# Build lookup: category_id -> category name
cat_map = {cat["id"]: cat["name"] for cat in coco_data["categories"]}

In [ ]:
def polygon_to_mask(polygon, height, width):
    """Convert COCO polygon segmentation (list of [x1,y1,x2,y2,...]) to a binary mask."""
    mask = np.zeros((height, width), dtype=np.uint8)
    pts = np.array(polygon, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)


def visualize_masks(image, masks, category_names=None):
    """Overlay segmentation masks from COCO annotations on an image."""
    # Composite masks directly onto image array to avoid darkening the background
    display = image.astype(np.float32) / 255.0
    alpha = 0.6
    for i, mask in enumerate(masks):
        color = np.random.rand(3,)
        for c in range(3):
            display[:, :, c] = np.where(
                mask,
                display[:, :, c] * (1 - alpha) + color[c] * alpha,
                display[:, :, c],
            )

    plt.figure(figsize=(10, 8))
    plt.imshow(display)

    for i, mask in enumerate(masks):
        if category_names:
            label = category_names[i] if i < len(category_names) else ""
            ys, xs = np.where(mask)
            if len(xs) > 0:
                cx, cy = int(xs.mean()), int(ys.mean())
                plt.text(cx, cy, label, color="white", fontsize=10,
                         bbox=dict(facecolor="black", alpha=0.6, boxstyle="round,pad=0.2"))
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def verify_annotation_for_image(img_path):
    """Load an image and overlay all its COCO segmentation masks."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    file_name = img_path.name
    img_id = image_id_map.get(file_name)
    if img_id is None:
        print(f"[SKIP] {file_name}: not found in annotations")
        return

    anns = anns_by_image_id.get(img_id, [])
    if not anns:
        print(f"[SKIP] {file_name}: no annotations")
        return

    masks = []
    category_names = []
    for ann in anns:
        for seg_poly in ann["segmentation"]:
            mask = polygon_to_mask(seg_poly, h, w)
            masks.append(mask)
            category_names.append(cat_map.get(ann["category_id"], str(ann["category_id"])))

    print(f"{file_name}: {len(anns)} annotations, {len(masks)} segments")
    visualize_masks(img, masks, category_names)


# Verify each sample image
for img_path in images_list:
    verify_annotation_for_image(img_path)